In [1]:
from pyspark.sql import SparkSession

# 1. Initialize Spark 
# (All Iceberg, S3, and JDBC configs are automatically inherited from spark-defaults.conf!)
spark = SparkSession.builder.appName("Iceberg-Lakehouse-Demo").getOrCreate()

# Silence verbose Spark/Iceberg warnings
spark.sparkContext.setLogLevel("ERROR")


26/09/13 16:25:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# 2. Create a Namespace (Schema)
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.training")
print("Namespace 'demo.training' ready.")

# 3. Create an Iceberg Table
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo.training.employee (
        emp_id BIGINT,
        emp_name STRING,
        department STRING,
        salary DOUBLE
    ) USING iceberg
""")
print("Iceberg table 'employee' ready.")

26/09/13 16:25:58 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1


Namespace 'demo.training' ready.
Iceberg table 'employee' ready.


In [3]:
spark.sql("Select * from demo.training.employee")

DataFrame[emp_id: bigint, emp_name: string, department: string, salary: double]

In [4]:
# 4. Write Data into the Table
spark.sql("""
    INSERT INTO demo.training.employee 
    VALUES 
        (1, 'Ashish Singh', 'Data Engineering', 150000.00),
        (2, 'Preeti Singh', 'Analytics', 135000.00)
""")
print("Data successfully committed to MinIO.")

# 5. Query the Active Data
print("\n--- CURRENT TABLE DATA ---")
spark.sql("SELECT * FROM demo.training.employee").show()

Data successfully committed to MinIO.

--- CURRENT TABLE DATA ---
+------+------------+----------------+--------+
|emp_id|    emp_name|      department|  salary|
+------+------------+----------------+--------+
|     1|Ashish Singh|Data Engineering|150000.0|
|     2|Preeti Singh|       Analytics|135000.0|
+------+------------+----------------+--------+



In [5]:

# ==========================================
# 6. VISUALIZE ICEBERG METADATA & HISTORY
# ==========================================

print("\n--- TABLE HISTORY (Timeline of Commits) ---")
# Shows when the table was created and every subsequent commit timestamp
spark.sql("SELECT * FROM demo.training.employee.history").show(truncate=False)

print("\n--- TABLE SNAPSHOTS ---")
# Shows the specific operations (e.g., 'append', 'overwrite') and row counts for each commit
spark.sql("SELECT snapshot_id, operation, summary FROM demo.training.employee.snapshots").show(truncate=False)

print("\n--- PHYSICAL DATA FILES ---")
# Exposes the actual Parquet files sitting in your MinIO 'warehouse' bucket
spark.sql("SELECT file_path, file_size_in_bytes, record_count FROM demo.training.employee.files").show(truncate=False)


--- TABLE HISTORY (Timeline of Commits) ---
+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-09-13 16:26:28.906|7001475892194027695|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+


--- TABLE SNAPSHOTS ---
+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |operation|summary                     

In [6]:
# Create a DataFrame
data = [(3, 'Ravi Kumar', 'Cloud Infrastructure', 1400200.00)]
columns = ["emp_id", "emp_name", "department", "salary"]
df = spark.createDataFrame(data, columns)

# Write to Iceberg using the V2 API
df.writeTo("demo.training.employee").append()

# Run the snapshots query again to see the new 'append' operation logged
spark.sql("SELECT snapshot_id, operation, summary FROM demo.training.employee.snapshots").show(truncate=False)

+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
+-------------------

In [6]:
spark.sql("SELECT * FROM demo.training.employee.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-09-03 17:02:53.083|2144457049189745365|NULL               |true               |
|2026-09-03 17:03:01.438|3568146871021356834|2144457049189745365|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [7]:
%%sql
    
SELECT * FROM demo.training.employee.history

26/09/03 17:03:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


made_current_at,snapshot_id,parent_id,is_current_ancestor
2026-09-03 17:02:53.083000,2144457049189745365,None,True
2026-09-03 17:03:01.438000,3568146871021356834,2144457049189745365,True


In [8]:
%%sql
SELECT * FROM demo.training.employee version as of 2144457049189745365;


emp_id,emp_name,department,salary
1,Ashish Singh,Data Engineering,150000.0
2,Preeti Singh,Analytics,135000.0
